<a href="https://colab.research.google.com/github/kamat-v/hedging-txtclf-experiments/blob/main/notebooks/distilbert_hedging_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## DistilBERT Baseline — Standalone Experiment
Standalone fine-tuning of **distilbert-base-uncased** on the real labeled dataset.
Purpose: validate dataset quality and establish transformer performance ceiling.
Not part of the main augmentation experimental pipeline.
Run on Google Colab with GPU.

Key results:
- Test F1 (threshold optimized): 0.887
- Test Precision: 0.808
- Test Recall: 0.984
- Overall soundness of data set; using frozen embeddings is CPU-feasible but a performance bottleneck in main experiments.

In [ ]:
%pip install transformers datasets venn-abers scikit-learn pyarrow -q

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, det_curve
from venn_abers import VennAbersCalibrator
import matplotlib.pyplot as plt
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
train_df = pd.read_parquet('/content/train.parquet')
cal_df = pd.read_parquet('/content/calibration.parquet')
test_df = pd.read_parquet('/content/test.parquet')

print(f"Train: {len(train_df)} | Positives: {train_df['label'].sum()}")
print(f"Cal:   {len(cal_df)} | Positives: {cal_df['label'].sum()}")
print(f"Test:  {len(test_df)} | Positives: {test_df['label'].sum()}")

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

class HedgeDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.encodings = tokenizer(
            df['sentence'].tolist(),
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(df['label'].values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

print("Building datasets...")
train_dataset = HedgeDataset(train_df, tokenizer)
cal_dataset = HedgeDataset(cal_df, tokenizer)
test_dataset = HedgeDataset(test_df, tokenizer)
print("Done.")

In [ ]:
# Compute class weights — consistent with class_weight='balanced' in sklearn
n_neg = (train_df['label'] == 0).sum()
n_pos = (train_df['label'] == 1).sum()
n_total = len(train_df)

weight_neg = n_total / (2 * n_neg)
weight_pos = n_total / (2 * n_pos)
class_weights = torch.tensor([weight_neg, weight_pos], dtype=torch.float).to(device)

print(f"Class weights — Negative: {weight_neg:.3f} | Positive: {weight_pos:.3f}")

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)
model.to(device)
print("Model loaded.")

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS = 3
BATCH_SIZE = 32
LR = 2e-5

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

print(f"Training for {EPOCHS} epochs, {len(train_loader)} steps per epoch...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 100 == 0:
            print(f"Epoch {epoch+1} | Step {batch_idx+1}/{len(train_loader)} | "
                  f"Loss: {total_loss/(batch_idx+1):.4f}")

    print(f"Epoch {epoch+1} complete | Avg loss: {total_loss/len(train_loader):.4f}")

print("Training complete.")

In [ ]:
def get_scores(model, dataset, batch_size=64):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size)
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels']

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)[:, 1]
            all_scores.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_scores), np.array(all_labels)

print("Extracting scores...")
y_scores_cal, y_cal = get_scores(model, cal_dataset)
y_scores_test, y_test = get_scores(model, test_dataset)
y_pred_test = (y_scores_test >= 0.5).astype(int)

print(f"Cal scores range:  [{y_scores_cal.min():.3f}, {y_scores_cal.max():.3f}]")
print(f"Test scores range: [{y_scores_test.min():.3f}, {y_scores_test.max():.3f}]")

In [ ]:
def optimal_threshold_f1(y_true, y_scores, thresholds=np.arange(0.01, 0.99, 0.01)):
    best_f1, best_t = 0, 0.5
    for t in thresholds:
        preds = (y_scores >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return best_t, best_f1

In [ ]:
# Raw at default threshold
print("=== DistilBERT — Test Set Raw (threshold=0.5) ===")
print(classification_report(y_test, y_pred_test, digits=3))

# Threshold optimized
t_opt, _ = optimal_threshold_f1(y_test, y_scores_test)
y_pred_opt = (y_scores_test >= t_opt).astype(int)
print(f"=== DistilBERT — Test Set Threshold Optimized (threshold={t_opt:.2f}) ===")
print(classification_report(y_test, y_pred_opt, digits=3))

In [ ]:
# Raw at default threshold
y_pred_cal_set = (y_scores_cal >= 0.5).astype(int)
print("=== DistilBERT — Calibration Set Raw (threshold=0.5) ===")
print(classification_report(y_cal, y_pred_cal_set, digits=3))

# Threshold optimized
t_cal_opt, _ = optimal_threshold_f1(y_cal, y_scores_cal)
y_pred_cal_opt = (y_scores_cal >= t_cal_opt).astype(int)
print(f"=== DistilBERT — Calibration Set Threshold Optimized (threshold={t_cal_opt:.2f}) ===")
print(classification_report(y_cal, y_pred_cal_opt, digits=3))

In [ ]:
fpr_raw, fnr_raw, _ = det_curve(y_test, y_scores_test)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_raw, fnr_raw, color='purple', linewidth=2)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('False Negative Rate', fontsize=12)
ax.set_title('DET Curve — DistilBERT Baseline (Raw Scores)', fontsize=13)
ax.grid(True, which='both', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('distilbert_baseline_det.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")